In [1]:
src_lang = "ind"
target_lang = ["aaz", "ptu", "nfa", "heg", "lex", "row", "llg", "rgu", "txq", "tet", "wrs"]
NT_BOOKS = [
    "MAT",
    "MRK",
    "LUK",
    "JHN",
    "ACT",
    "ROM",
    "1CO",
    "2CO",
    "GAL",
    "EPH",
    "PHP",
    "COL",
    "1TH",
    "2TH",
    "1TI",
    "2TI",
    "TIT",
    "PHM",
    "HEB",
    "JAS",
    "1PE",
    "2PE",
    "1JN",
    "2JN",
    "3JN",
    "JUD",
    "REV",
]

In [10]:
from datasets import load_dataset

dataset = load_dataset("DavidCBaines/ebible_corpus", split="train", num_proc=10)
print(dataset)
count = 0
for datum in dataset:
    if datum["ind"]:
        print(f"Book: {datum['book']} {datum['chapter']}:{datum['verse']}")
        print(f"indags: {datum['indags']}")
    count += 1
    if count > 100:
        break

Dataset({
    features: ['book', 'chapter', 'verse', 'aai', 'aak', 'aau', 'aaz', 'abc', 'abt-maprik', 'abt-wosera', 'abx', 'aby', 'acfNT', 'acr-acc', 'acrNNT', 'acrTNT', 'acuNT', 'adz', 'aer', 'aey', 'agd', 'agg', 'agm', 'agn', 'agr', 'agt', 'aguBl', 'ahr', 'aia', 'aii', 'akeNT', 'alpNT', 'alqALGNT', 'alw', 'aly', 'ameNT', 'amf', 'amh', 'amk', 'amm', 'amn-amanab', 'amn-n', 'amo', 'amp', 'amrNT', 'amuNT', 'amx', 'anh', 'anvNT', 'aoi', 'aoj', 'aoj-filifita', 'aom', 'aon', 'apb', 'apeB', 'apec', 'apnNT', 'apr', 'apuNT', 'apwNT', 'apz', 'arb-vd', 'arbnav', 'are', 'arlNT', 'arnNT', 'arp', 'asj', 'asmfb', 'aso', 'ata', 'atbNT', 'atdNT', 'atgNT', 'att', 'aucNT', 'aui', 'auy', 'avt', 'awb', 'awk', 'awx', 'azb', 'azgNT', 'azzNT', 'baoNT', 'bba', 'bbb', 'bbr', 'bbr2013', 'bch', 'bco', 'bdd', 'bdv', 'bea', 'bef', 'bel', 'beln', 'benirv', 'benobcv', 'beo', 'beu', 'bfz', 'bgc', 'bgg', 'bgs', 'bgt', 'bhd', 'bhg', 'bhi', 'bhl', 'bht', 'bhu', 'big', 'big2013', 'bjk', 'bjp', 'bjr', 'bjvNT', 'bjz', 'bkd

In [1]:
from datasets import load_dataset

# dataset = load_dataset("bible-nlp/biblenlp-corpus", languages=["hau", "daa"], pair="range", trust_remote_code=True)
dataset = load_dataset("bible-nlp/biblenlp-corpus", languages=["aaz"], trust_remote_code=True)

print(dataset)

/data/projects/punim0478/setiawand/.cache/conda-envs/david/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights'],
        num_rows: 289
    })
    validation: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights'],
        num_rows: 15
    })
    test: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights'],
        num_rows: 19
    })
})


In [32]:
for datum in dataset["train"]:
    if "GEN 1:1" in datum["ref"][0]:
        print(datum["ref"])
        print(datum)
        # print(datum["files"])
        # print(datum["translation"]["translation"])
    # if len(datum["ref"]) > 1:
    #     print(f"Ref: {datum['ref']}")
    #     print(f"Files: {datum['files']}")
    #     print(f"Licenses: {datum['licenses']}")
    #     print(f"Language: {datum['translation']['language']}")
    #     print(f"Translation: {datum['translation']['translation']}")
    #     print(f"Translation length: {len(datum['translation']['translation'])}\n")
        
        # break

['GEN 1:14', 'GEN 1:15']
{'translation': {'language': ['ind'], 'translation': ['berkatalah Allah, “Hendaklah ada berbagai benda penerang di langit supaya sinarnya terpancar ke bumi. Biarlah benda-benda itu menunjukkan perbedaan antara siang dan malam, dan menjadi tanda untuk menentukan hari, tahun, dan musim.” Maka jadilah demikian.']}, 'files': {'lang': ['ind'], 'file': ['ind-ind.txt']}, 'ref': ['GEN 1:14', 'GEN 1:15'], 'licenses': ['http://creativecommons.org/licenses/by-nd/4.0/'], 'copyrights': ['oleh Yayasan Alkitab BahasaKita (Albata)']}


In [30]:
from datasets import load_dataset, DatasetDict, concatenate_datasets

def process_translations(x, src_lang: str, tgt_lang: str):
    """
    Select a single source/target translation per row and record chosen files.
    - Source is `src_lang`; if multiple versions exist, prefer '{src_lang}-{src_lang}.txt' (e.g., 'ind-ind.txt').
    - Target is `tgt_lang`; assumed unique in the row.
    Returns: text_source, text_target, source_file, target_file
    """
    tr = x.get("translation") or {}
    languages = list(tr.get("language") or [])
    texts = list(tr.get("translation") or [])

    files_info = x.get("files") or {}
    file_names = files_info.get("file") if isinstance(files_info, dict) else None

    # Select source (prefer '{src}-{src}.txt' when multiple)
    src_indices = [i for i, lang in enumerate(languages) if lang == src_lang]
    selected_source_idx = None
    if src_indices:
        if len(src_indices) == 1:
            selected_source_idx = src_indices[0]
        else:
            preferred_idx = None
            if isinstance(file_names, list) and len(file_names) == len(languages):
                preferred_filename = f"{src_lang}-{src_lang}.txt"
                for idx in src_indices:
                    if file_names[idx] == preferred_filename:
                        preferred_idx = idx
                        break
            selected_source_idx = preferred_idx if preferred_idx is not None else src_indices[0]

    # Select target (first occurrence of tgt_lang)
    tgt_indices = [i for i, lang in enumerate(languages) if lang == tgt_lang]
    selected_target_idx = tgt_indices[0] if tgt_indices else None

    source_text = texts[selected_source_idx] if selected_source_idx is not None and selected_source_idx < len(texts) else ""
    target_text = texts[selected_target_idx] if selected_target_idx is not None and selected_target_idx < len(texts) else ""

    source_file = ""
    target_file = ""
    if isinstance(file_names, list) and len(file_names) == len(languages):
        if selected_source_idx is not None and selected_source_idx < len(file_names):
            source_file = file_names[selected_source_idx]
        if selected_target_idx is not None and selected_target_idx < len(file_names):
            target_file = file_names[selected_target_idx]

    return {
        "text_source": source_text,
        "text_target": target_text,
        "source_file": source_file,
        "target_file": target_file,
    }
    

def load_ebible_corpus(src_lang, tgt_lang):
    dataset = load_dataset("bible-nlp/biblenlp-corpus", languages=[src_lang, tgt_lang], trust_remote_code=True)
    dataset = dataset.map(process_translations, fn_kwargs={"src_lang": src_lang, "tgt_lang": tgt_lang})
    # OT books for testing, NT books for training and validation
    # Handle both single refs and multiple refs
    def is_nt_book(refs):
        if isinstance(refs, list):
            # Check if any ref belongs to NT books
            return any(ref.split()[0] in NT_BOOKS for ref in refs)
        else:
            # Single reference
            return refs.split()[0] in NT_BOOKS
    
    # The dataset is a DatasetDict, so we need to access the 'train' split
    train_data = dataset['train']
    valid_data = dataset['validation']
    train_data = concatenate_datasets([train_data, valid_data])
    train_ds = train_data.filter(lambda x: is_nt_book(x["ref"]))
    test_ds = train_data.filter(lambda x: not is_nt_book(x["ref"]))
    train_val_ds = train_ds.train_test_split(test_size=0.05, seed=41)
    dataset = DatasetDict({"train": train_val_ds["train"], "validation": train_val_ds["test"], "test": test_ds})
    return dataset

dataset = load_ebible_corpus("hau", "daa")

In [31]:
dataset

DatasetDict({
    train: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights', 'text_source', 'text_target', 'source_file', 'target_file'],
        num_rows: 7132
    })
    validation: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights', 'text_source', 'text_target', 'source_file', 'target_file'],
        num_rows: 376
    })
    test: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights', 'text_source', 'text_target', 'source_file', 'target_file'],
        num_rows: 0
    })
})

In [12]:
from transformers import AutoTokenizer

split = ["train", "validation", "test"]
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
# tokenizer.add_special_tokens({"additional_special_tokens": ["wrs_Latn"]})
tokenizer.src_lang = "ksd"
tokenizer.tgt_lang = "kqw"
max_input_len = 0
max_label_len = 0

for s in split:
    dataset_split = dataset[s]
    for datum in dataset_split:
        tokens = tokenizer(
                    datum["text_source"],
                    text_target=datum["text_target"],
                    max_length=1024,
                )
        max_input_len = max(max_input_len, len(tokens["input_ids"]))
        max_label_len = max(max_label_len, len(tokens["labels"]))
print("max input len:", max_input_len)
print("max label len:", max_label_len)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


max input len: 93
max label len: 196


In [13]:
print(tokenizer.aaz_Latn)

aaz


In [20]:
for datum in dataset["train"]:
    if "GEN" in datum["ref"][0]:
        print(datum["ref"])
    
        
    # if len(datum["ref"]) > 1:
    #     print(f"Ref: {datum['ref']}")
    #     print(f"Files: {datum['files']}")
    #     print(f"Licenses: {datum['licenses']}")
    #     print(f"Language: {datum['translation']['language']}")
    #     print(f"Translation: {datum['translation']['translation']}")
        # print(f"Source file: {datum['source_file']}")
        # print(f"Target file: {datum['target_file']}")
        # print(f"Source: {datum['text_source']}")
        # print(f"Target: {datum['text_target']}\n")
        
        # break

['GEN 49:29', 'GEN 49:30']
['GEN 49:14', 'GEN 49:15']
['GEN 47:1', 'GEN 47:2']
['GEN 46:6', 'GEN 46:7']
['GEN 44:30', 'GEN 44:31']
['GEN 43:21', 'GEN 43:22']
['GEN 43:8', 'GEN 43:9']
['GEN 42:17', 'GEN 42:18']
['GEN 42:7', 'GEN 42:8']
['GEN 41:41', 'GEN 41:42']
['GEN 41:30', 'GEN 41:31']
['GEN 40:1', 'GEN 40:2', 'GEN 40:3']
['GEN 39:14', 'GEN 39:15']
['GEN 39:1', 'GEN 39:2']
['GEN 38:18', 'GEN 38:19']
['GEN 37:5', 'GEN 37:6', 'GEN 37:7']
['GEN 36:40', 'GEN 36:41', 'GEN 36:42', 'GEN 36:43']
['GEN 36:29', 'GEN 36:30']
['GEN 36:25', 'GEN 36:26']
['GEN 36:20', 'GEN 36:21']
['GEN 36:15', 'GEN 36:16']
['GEN 36:10', 'GEN 36:11', 'GEN 36:12', 'GEN 36:13']
['GEN 36:6', 'GEN 36:7', 'GEN 36:8']
['GEN 36:4', 'GEN 36:5']
['GEN 36:2', 'GEN 36:3']
['GEN 35:28', 'GEN 35:29']
['GEN 35:13', 'GEN 35:14']
['GEN 35:2', 'GEN 35:3']
['GEN 32:20', 'GEN 32:21']
['GEN 31:51', 'GEN 31:52']
['GEN 31:24', 'GEN 31:25']
['GEN 31:17', 'GEN 31:18', 'GEN 31:19', 'GEN 31:20']
['GEN 31:14', 'GEN 31:15']
['GEN 29:23', 'GE

In [14]:
# Push the combined dataset to Hugging Face Hub
def push_to_hub(dataset, repo_name, private=False):
    """
    Push the processed dataset to Hugging Face Hub
    
    Args:
        dataset: The DatasetDict to push
        repo_name: Name of the repository (e.g., "username/dataset-name")
        private: Whether to make the repository private
    """
    try:
        # Push the dataset
        dataset.push_to_hub(
            repo_id=repo_name,
            private=private,
            token=True  # Uses your saved HF token
        )
        print(f"✅ Successfully pushed dataset to: https://huggingface.co/datasets/{repo_name}")
        
        # Print dataset card information
        print(f"\n📝 Dataset structure:")
        for subset_name in dataset.keys():
            lang_pair = subset_name.rsplit('-', 1)[0]  # Remove split suffix
            split = subset_name.rsplit('-', 1)[1]      # Get split name
            print(f"  {subset_name}: {len(dataset[subset_name])} examples")
            
    except Exception as e:
        print(f"❌ Error pushing to hub: {e}")
        print("Make sure you're logged in with `huggingface-cli login`")

# Example usage (uncomment and modify as needed):
# REPO_NAME = "your-username/bible-nmt-multilingual"  # Change this to your desired repo name
# push_to_hub(combined_dataset, REPO_NAME, private=False)

print("Dataset is ready to be pushed to Hub!")
print("Uncomment and modify the REPO_NAME above, then run the push_to_hub function.")


Dataset is ready to be pushed to Hub!
Uncomment and modify the REPO_NAME above, then run the push_to_hub function.


In [15]:
combined_dataset.push_to_hub("biblenlp-corpus")

ValueError: Split name should match '^\w+(\.\w+)*$' but got 'ind-wrs-train'.